In [2]:
import sys

In [ ]:
#!{sys.executable} -m pip install langchain-openai langchain-community
#!{sys.executable} -m pip install langchain-qdrant

  Using cached langchain_qdrant-1.1.0-py3-none-any.whl.metadata (2.0 kB)
Using cached langchain_qdrant-1.1.0-py3-none-any.whl (24 kB)

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


In [3]:
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
#from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from qdrant_client.models import Distance, VectorParams
from qdrant_client import QdrantClient
from dotenv import load_dotenv
import pandas as pd

In [4]:
load_dotenv("si699.env")

True

## Part 1 Chunking

In [23]:
damage_df = pd.read_json("dnd5eapi_dump/damage-types.json")
magic_schools_df = pd.read_json("dnd5eapi_dump/magic-schools.json")
alignments_df = pd.read_json("dnd5eapi_dump/alignments.json")
skills_df = pd.read_json("dnd5eapi_dump/skills.json")
languages_df = pd.read_json("dnd5eapi_dump/languages.json")
weapon_df = pd.read_json("dnd5eapi_dump/weapon-properties.json")
conditions_df = pd.read_json("dnd5eapi_dump/conditions.json")
equipment_df = pd.read_json("dnd5eapi_dump/equipment.json")
feats_df = pd.read_json("dnd5eapi_dump/feats.json")
features_df = pd.read_json("dnd5eapi_dump/features.json")
magic_items_df = pd.read_json("dnd5eapi_dump/magic-items.json")
traits_df = pd.read_json("dnd5eapi_dump/traits.json")
proficiencies_df = pd.read_json("dnd5eapi_dump/proficiencies.json")
equipment_categories_df = pd.read_json("dnd5eapi_dump/equipment-categories.json")

### Helper function

In [25]:
def join_desc(value):
    if value is None:
        return ""
    if isinstance(value, list):
        return "\n".join(str(x) for x in value if x)
    return str(value)

def ref_name(obj, default=""):
    if isinstance(obj, dict):
        return obj.get("name", default)
    return default

def ref_names(items):
    if not items:
        return ""
    return ", ".join(
        x.get("name", "") for x in items
        if isinstance(x, dict) and x.get("name")
    )

def make_chunk(chunk_id, text, metadata):
    return {
        "id": chunk_id,
        "text": text.strip(),
        "metadata": metadata
    }

def base_metadata(endpoint, item, chunk_type="main", chunk_order=0):
    return {
        "endpoint": endpoint,
        "index": item.get("index"),
        "name": item.get("name"),
        "url": item.get("url"),
        "updated_at": item.get("updated_at"),
        "chunk_type": chunk_type,
        "chunk_order": chunk_order
    }

### Chunking function for small content size document.

In [26]:
def create_chunk(endpoint, url, items):
    desc = items.get("desc", "")
    if isinstance(desc, list):
        desc = " ".join(desc)
    ability = items.get("ability_score", {}).get("name", None)
    text = f"""
Type: {endpoint.replace('-', ' ').title()}
Name: {items.get("name")}
Associated Ability: {ability}

Category: {items.get("type", None)}
Script: {items.get("script", None)}
Typical Speakers: {", ".join(items.get("typical_speakers", []))}

Description:
{desc}
""".strip()

    chunk = {
        "id": f"{endpoint}_{items['index']}",
        "text": text,
        "metadata": {
            "endpoint": endpoint,
            "name": items.get("name"),
            "index": items.get("index"),
            "url": url + items.get("index"),
            "updated_at": items.get("updated_at"),
            "category": endpoint
        }
    }

    return chunk

In [27]:
def build_chunks(df):
    chunks = []
    for i in range(len(df)):
        endpoint, url, items = df.iloc[i, :]
        chunk = create_chunk(endpoint, url, items)
        chunks.append(chunk)
    return chunks

In [28]:
chunks = build_chunks(damage_df)
chunks.extend(build_chunks(magic_schools_df))
chunks.extend(build_chunks(alignments_df))
chunks.extend(build_chunks(skills_df))
chunks.extend(build_chunks(languages_df))
chunks.extend(build_chunks(weapon_df))
chunks.extend(build_chunks(conditions_df))

### chunking function for medium context size file

In [29]:
def chunk_feats_df(df: pd.DataFrame) -> list[dict]:
    chunks = []

    for _, row in df.iterrows():
        endpoint = row["endpoint"]
        item = row["items"]

        prereq_lines = []
        for p in item.get("prerequisites", []):
            ability = ref_name(p.get("ability_score"))
            minimum = p.get("minimum_score")
            if ability and minimum is not None:
                prereq_lines.append(f"{ability} {minimum}")
            else:
                prereq_lines.append(str(p))

        prereq_text = ", ".join(prereq_lines) if prereq_lines else "None"
        desc_text = join_desc(item.get("desc"))

        text = f"""
Type: Feat
Name: {item.get("name")}

Prerequisites: {prereq_text}

Description:
{desc_text}
"""

        chunks.append(
            make_chunk(
                chunk_id=f"{endpoint}_{item.get('index')}_main",
                text=text,
                metadata=base_metadata(endpoint, item, "main", 0)
            )
        )

    return chunks

In [30]:
def chunk_traits_df(df: pd.DataFrame) -> list[dict]:
    chunks = []

    for _, row in df.iterrows():
        endpoint = row["endpoint"]
        item = row["items"]

        races = ref_names(item.get("races", []))
        subraces = ref_names(item.get("subraces", []))
        profs = ref_names(item.get("proficiencies", []))
        desc_text = join_desc(item.get("desc"))

        text = f"""
Type: Trait
Name: {item.get("name")}

Races: {races if races else "None"}
Subraces: {subraces if subraces else "None"}
Related Proficiencies: {profs if profs else "None"}

Description:
{desc_text}
"""

        chunks.append(
            make_chunk(
                chunk_id=f"{endpoint}_{item.get('index')}_main",
                text=text,
                metadata=base_metadata(endpoint, item, "main", 0)
            )
        )

    return chunks

In [31]:
def chunk_proficiencies_df(df: pd.DataFrame) -> list[dict]:
    chunks = []

    for _, row in df.iterrows():
        endpoint = row["endpoint"]
        item = row["items"]

        classes = ref_names(item.get("classes", []))
        races = ref_names(item.get("races", []))
        ref_obj = item.get("reference", {})

        text = f"""
Type: Proficiency
Name: {item.get("name")}
Category: {item.get("type", "")}

Classes: {classes if classes else "None"}
Races: {races if races else "None"}
Reference: {ref_name(ref_obj)}

Description:
This proficiency belongs to the category "{item.get("type", "")}" and references "{ref_name(ref_obj)}".
"""

        chunks.append(
            make_chunk(
                chunk_id=f"{endpoint}_{item.get('index')}_main",
                text=text,
                metadata=base_metadata(endpoint, item, "main", 0)
            )
        )

    return chunks

In [32]:
def chunk_equipment_df(df: pd.DataFrame) -> list[dict]:
    chunks = []

    for _, row in df.iterrows():
        endpoint = row["endpoint"]
        item = row["items"]

        eq_cat = ref_name(item.get("equipment_category"))
        gear_cat = ref_name(item.get("gear_category"))

        cost = item.get("cost", {})
        cost_text = ""
        if isinstance(cost, dict) and cost.get("quantity") is not None and cost.get("unit"):
            cost_text = f"{cost['quantity']} {cost['unit']}"

        overview_text = f"""
Type: Equipment
Name: {item.get("name")}

Equipment Category: {eq_cat if eq_cat else "Unknown"}
Gear Category: {gear_cat if gear_cat else "None"}
Cost: {cost_text if cost_text else "Unknown"}
Weight: {item.get("weight", "Unknown")}
"""

        chunks.append(
            make_chunk(
                chunk_id=f"{endpoint}_{item.get('index')}_overview",
                text=overview_text,
                metadata=base_metadata(endpoint, item, "overview", 0)
            )
        )

        desc_text = join_desc(item.get("desc"))
        special_text = join_desc(item.get("special"))
        contents = item.get("contents", [])
        properties = item.get("properties", [])

        detail_parts = []

        if desc_text:
            detail_parts.append(f"Description:\n{desc_text}")

        if special_text:
            detail_parts.append(f"Special:\n{special_text}")

        if contents:
            content_lines = []
            for c in contents:
                if isinstance(c, dict):
                    name = ref_name(c.get("item"))
                    qty = c.get("quantity")
                    if name:
                        content_lines.append(f"{name} x{qty}" if qty is not None else name)
            if content_lines:
                detail_parts.append("Contents:\n" + "\n".join(content_lines))

        if properties:
            prop_names = ref_names(properties)
            if prop_names:
                detail_parts.append(f"Properties:\n{prop_names}")

        if detail_parts:
            detail_text = f"""
Type: Equipment
Name: {item.get("name")}

""" + "\n\n".join(detail_parts)

            chunks.append(
                make_chunk(
                    chunk_id=f"{endpoint}_{item.get('index')}_details",
                    text=detail_text,
                    metadata=base_metadata(endpoint, item, "details", 1)
                )
            )

    return chunks

In [33]:
def chunk_features_df(df: pd.DataFrame) -> list[dict]:
    chunks = []

    for _, row in df.iterrows():
        endpoint = row["endpoint"]
        item = row["items"]

        class_name = ref_name(item.get("class"))
        subclass_name = ref_name(item.get("subclass"))
        prereqs = item.get("prerequisites", [])
        prereq_text = ", ".join(str(x) for x in prereqs) if prereqs else "None"
        desc_text = join_desc(item.get("desc"))

        text = f"""
Type: Class Feature
Name: {item.get("name")}

Class: {class_name if class_name else "Unknown"}
Subclass: {subclass_name if subclass_name else "None"}
Level: {item.get("level", "Unknown")}
Prerequisites: {prereq_text}

Description:
{desc_text}
"""

        chunks.append(
            make_chunk(
                chunk_id=f"{endpoint}_{item.get('index')}_main",
                text=text,
                metadata=base_metadata(endpoint, item, "main", 0)
            )
        )

    return chunks

In [34]:
def chunk_magic_items_df(df: pd.DataFrame) -> list[dict]:
    chunks = []

    for _, row in df.iterrows():
        endpoint = row["endpoint"]
        item = row["items"]

        eq_cat = ref_name(item.get("equipment_category"))
        rarity = ref_name(item.get("rarity"))
        variant_flag = item.get("variant", False)

        overview_text = f"""
Type: Magic Item
Name: {item.get("name")}

Equipment Category: {eq_cat if eq_cat else "Unknown"}
Rarity: {rarity if rarity else "Unknown"}
Variant Item: {"Yes" if variant_flag else "No"}
"""

        chunks.append(
            make_chunk(
                chunk_id=f"{endpoint}_{item.get('index')}_overview",
                text=overview_text,
                metadata=base_metadata(endpoint, item, "overview", 0)
            )
        )

        desc_text = join_desc(item.get("desc"))
        variants = ref_names(item.get("variants", []))
        image = item.get("image", "")

        detail_parts = []
        if desc_text:
            detail_parts.append(f"Description:\n{desc_text}")
        if variants:
            detail_parts.append(f"Variants:\n{variants}")
        if image:
            detail_parts.append(f"Image Path:\n{image}")

        if detail_parts:
            detail_text = f"""
Type: Magic Item
Name: {item.get("name")}

""" + "\n\n".join(detail_parts)

            chunks.append(
                make_chunk(
                    chunk_id=f"{endpoint}_{item.get('index')}_details",
                    text=detail_text,
                    metadata=base_metadata(endpoint, item, "details", 1)
                )
            )

    return chunks

In [35]:
def chunk_equipment_categories_df(df: pd.DataFrame) -> list[dict]:
    chunks = []

    for _, row in df.iterrows():
        endpoint = row["endpoint"]
        item = row["items"]

        equipment_names = ref_names(item.get("equipment", []))

        text = f"""
Type: Equipment Category
Name: {item.get("name")}

Equipment in this Category:
{equipment_names if equipment_names else "None"}
"""

        chunks.append(
            make_chunk(
                chunk_id=f"{endpoint}_{item.get('index')}_main",
                text=text,
                metadata=base_metadata(endpoint, item, "main", 0)
            )
        )

    return chunks

In [ ]:
chunks.extend(chunk_feats_df(feats_df))
chunks.extend(chunk_features_df(features_df))
chunks.extend(chunk_magic_items_df(magic_items_df))
chunks.extend(chunk_proficiencies_df(proficiencies_df))
chunks.extend(chunk_traits_df(traits_df))
chunks.extend(chunk_equipment_df(equipment_df))
chunks.extend(chunk_equipment_categories_df(equipment_categories_df))

TypeError: 'function' object is not iterable

## Part2: Embedding and Store Vectors

In [38]:
# model initialization
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
client = QdrantClient(path="./qdrant_local") # specify the path so that it persist locally on disk

docs = []
for chunk in chunks:
    docs.append(Document(
        page_content=chunk["text"],
        metadata = {
            **chunk["metadata"],
            "chunk_id": chunk["id"]
        }
    ))

In [39]:
collection_name = "damage_types"

client.create_collection(
    collection_name= collection_name,
    vectors_config= VectorParams(size=1536, #dimensinality of the vector
                                 distance=Distance.COSINE # distance metric for similarity search
    )
)

vectorstore = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embedding_model
)

vectorstore.add_documents(docs)


['b7d7a756a2ff4472804a33e0ea560f7d',
 'e3f652f5dcb941319a4cb0011ad4d6e4',
 'dca3d6976c894522b42402b69aa9c50c',
 'c3c7fa7418224216a939849890cb3feb',
 '7ba367778b4e449f82e1173ff6ce5291',
 '01cd1276c1194112ac66d65cc51d597a',
 '9bbab224ee294bef903fc126c3d231d3',
 '5c28188a969644769742a91e3fcd8d75',
 '835e704b39a14cf3b4b96c058dc1f9eb',
 '702d3cdb8b194d82866c380523467ffa',
 'c6da00c956284091942b77f4d194e9ce',
 '1ef20240c0234877b809d0fc77c9397e',
 '498c4fbebba2413985ec5e5d8e4abfb2',
 'a6b7dbfd5eaf4ce6b8cdcce87ef74024',
 '89ef354a2c58493a8f76535b16d27fe2',
 '513487ae66844fc4a807bbba4f1d2c66',
 '3b8ef15a6bf544bdbf2d8ac015522390',
 '0f201663007d418a95cc99f97764b27a',
 '8ae8178e3e2e454ab6b2920cd14b2e82',
 '95992da1012041aa9c6e89643d626e0b',
 '80435a23662c4857a6bebdbf5160bd21',
 '496c2d414d0f47179ea02973731e8706',
 'e2389306d8004c2e956d1788a3e932fe',
 '5d15f54e08ab4fc4acb45e0c84e30c92',
 'd3f7c374705b43ac8645544b2bc17174',
 'b57f496daaa344ec84a5bdc1145f8c3f',
 '6fcc9c0b43a04bf69f47733a1c93923b',
 

## Part 3: Retrival

In [40]:
def retrieve_chunks(vectorstore, query, k=3):
    results = vectorstore.similarity_search(query, k=k)
    return results

In [42]:
def build_context(retrieved_docs):
    contexts = []
    for i, doc in enumerate(retrieved_docs, 1):
        contexts.append(f"[Chunk {i} | {doc.metadata.get('chunk_id')} | {doc.metadata.get('name')}]\n{doc.page_content}")
        
    return "\n\n".join(contexts)

In [43]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
rag_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are a helpful Dungeons & Dragons rules assistant.

Use only the retrieved context below to answer the question.
If the answer is not contained in the context, say "I don't know based on the retrieved context."

Retrieved Context:
{context}

Question:
{question}

Answer:
""".strip()
)

In [46]:
def rag_pipeline(query, llm, vectorstore, prompt_template, k=3):
    retrived_docs = retrieve_chunks(vectorstore, query, k)
    context = build_context(retrived_docs)
    prompt = prompt_template.format(context=context, question=query)
    answer = llm.invoke(prompt).content
    
    return retrived_docs, context, answer

In [55]:
query = "what gear cost 39gp?"

retrieved_docs, context, answer = rag_pipeline(
    query=query,
    vectorstore=vectorstore,
    llm=llm,
    prompt_template=rag_prompt,
    k=15
)

print(context)
print()
print(answer)

[Chunk 1 | equipment_clothes-fine_overview | Clothes, fine]
Type: Equipment
Name: Clothes, fine

Equipment Category: Adventuring Gear
Gear Category: Standard Gear
Cost: 15 gp
Weight: 6

[Chunk 2 | equipment_clothes-costume_overview | Clothes, costume]
Type: Equipment
Name: Clothes, costume

Equipment Category: Adventuring Gear
Gear Category: Standard Gear
Cost: 5 gp
Weight: 4

[Chunk 3 | equipment_clothes-common_overview | Clothes, common]
Type: Equipment
Name: Clothes, common

Equipment Category: Adventuring Gear
Gear Category: Standard Gear
Cost: 5 sp
Weight: 3

[Chunk 4 | equipment_studded-leather-armor_overview | Studded Leather Armor]
Type: Equipment
Name: Studded Leather Armor

Equipment Category: Armor
Gear Category: None
Cost: 45 gp
Weight: 13

[Chunk 5 | equipment_hourglass_overview | Hourglass]
Type: Equipment
Name: Hourglass

Equipment Category: Adventuring Gear
Gear Category: Standard Gear
Cost: 25 gp
Weight: 1

[Chunk 6 | equipment_scholars-pack_overview | Scholar's Pack]
